In [6]:
import re
import json
import joblib
from modules import spark_init
from modules import network_constructor as nc

data_path = "data_shared/reddit"

# Bot Filtering

We filter authors that have high probability of being bots if:

- Authors posted on more than 50 distinct subreddits for at least one month.

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12

#Here we define the paths to the data
years = [str(year) for year in range(min_year, max_year+1)]
months = [str(month).zfill(2) for month in range(1, 13)]
paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" for year in years for month in months]

#Here we read the data
processed_bots = []
for timestep, path in enumerate(paths):
    rdd = sc.textFile(path)
    bots = (
        rdd
        .map(json.loads) #Load the JSON
        .map(lambda x: (x.get("author"), x.get("subreddit"))) #Get the necessary fields
        .distinct() #Get the unique authors and subreddits pairs
        .map(lambda x: (x[0], 1)) #((author, 1))
        .reduceByKey(lambda x, y: x + y) #Count the number of unique subreddits per author
        .filter(lambda x: x[1] > 50) #Filter out authors with less or equal than 50 different subreddits
        .map(lambda x: x[0]) #Get the author
        )
    processed_bots.append(bots)

#Here we concatenate the RDDs
bots_above_threshold = sc.union(processed_bots).distinct().collect()
#Here we save the RDD using joblib
joblib.dump(bots_above_threshold, f"{data_path}/new_bot_list.joblib")
#Here we stop the Spark context
sc.stop()

We also filter authors that have high probability of being bots if:

- They contain the word `bot` in their usernames (except common dictionary words).

In [3]:
def bot_search(username):

    bot_rex = re.compile(r'bot', re.IGNORECASE)
    non_bot_words = ['both', 'bottom', 'bottle', 'botany', 'botanist', 'botanic', 'botched']
    
    if type(username) != str:
        return(False)
    
    c1 = bot_rex.search(username) is not None
    if not c1:
        return False
    
    c2 = False
    for j in non_bot_words:
        if re.search(j, username, re.IGNORECASE):
            c2 = True
            break

    if c1 and not c2:
        return True
    else: return False

In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12

#Here we define the paths to the data
years = [str(year) for year in range(min_year, max_year+1)]
months = [str(month).zfill(2) for month in range(1, 13)]
paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" for year in years for month in months]

#Here we read all the data in a single RDD
rdd = sc.textFile(','.join(paths))
#Here we filter out the bots
string_bots = (
    rdd
    .map(json.loads) #Load the JSON
    .map(lambda x: x.get("author")) #Get the author
    .filter(bot_search) #Filter out the bots
    .distinct() #Get the unique authors
    .collect() #Collect the results
)
#Here we save the RDD using joblib
joblib.dump(string_bots, f"{data_path}/string_bot_list.joblib")
#Here we stop the Spark context
sc.stop()

# Subreddit Filtering

We exclude subreddits by the following criteria:

- Selecting only those that have at least one month with at least 50 distinct users publishing a post on it 

- Selecting only those that belong to the Demographics study.


In [ ]:
#Here we initialize the Spark context
sc = spark_init.spark_context()

#Here we read the minimum year and maximum year as arguments
min_year = 2019
max_year = 2023
total_months = (max_year - min_year + 1) * 12

#Here we define the paths to the data
years = [str(year) for year in range(min_year, max_year+1)]
months = [str(month).zfill(2) for month in range(1, 13)]
paths = [f"{data_path}/submissions/{year}/RS_{year}-{month}.bz2" for year in years for month in months]


#Here we read the data
processed_subreddits = []
for timestep, path in enumerate(paths):
    rdd = sc.textFile(path)
    subreddits = (
        rdd
        .map(json.loads) #Load the JSON
        .map(lambda x: (x.get("subreddit"), x.get("author"))) #Get the necessary fields
        .distinct() #Get the unique authors and subreddits pairs
        .map(lambda x: (x[0], 1)) #((subreddit, 1))
        .reduceByKey(lambda x, y: x + y) #Count the number of unique authors per subreddit
        .filter(lambda x: x[1] >= 50) #Filter out subreddits with less than 50 different authors
        .map(lambda x: x[0]) #Get the subreddit
        )
    processed_subreddits.append(subreddits)

#Here we concatenate the RDDs
full_subreddits = sc.union(processed_subreddits).distinct().collect()
#Here we save the RDD using joblib
joblib.dump(full_subreddits, f"{data_path}/subreddit_list.joblib")
#Here we stop the Spark context
sc.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
